# SOURCE REGISTRY EDA

This EDA works for assigning the PERMISSIONS.md candidates

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from evidence_retrieval_co.paths import REGISTRY_CSV

pd.set_option('display.max_columns', None)

In [ ]:
# Load the csv and basic info

df = pd.read_csv(REGISTRY_CSV)

print(f"Data shape is {df.shape[0]} rows and {df.shape[1]} columns")

print(f"Columns are: {df.columns.tolist()}")

df.head()

In [ ]:
# Knos the dtype and missing values of the columns

df.info()

No missing values to deal with

In [ ]:
# Pareto rule 

press = df[df["tier"] == "press"].sort_values(by="n_articles", ascending=False)

press["articles_cumsum"] = press["n_articles"].cumsum()

articles_cumsum = press["articles_cumsum"].iloc[-1] # Last value of the cumulative sum

press["articles_percentage"] = press["articles_cumsum"] / articles_cumsum * 100

pareto_prensa = press[press["articles_percentage"].shift(1, fill_value=0) < 80]

filter = (df["scope_flag_partisan"] == True) & (df["tier"] == "press")
hue_filter = "scope_flag_partisan" if filter.any() else "n_articles"

plt.figure(figsize=(10, 6))
sns.barplot(data=pareto_prensa, x="n_articles", y="domain", palette="viridis", hue=hue_filter, dodge=False, legend=False)
bars = plt.gca().patches
for bar in bars:
    width = bar.get_width()
    plt.gca().text(width + 1, bar.get_y() + bar.get_height()/2, f'{int(width)}', va='center', fontsize=8)

plt.annotate(
    text=f"80% of articles are from\n{pareto_prensa.shape[0]} sources of {press.shape[0]} total sources",
    xy=(0.7, 0.5),
    xycoords="axes fraction",
    fontsize=10,
    bbox={'boxstyle': "round,pad=0.3", 'fc': "yellow", 'alpha': 0.5}
)
plt.title("Paretto Rule: Press Sources with 80% of Mentions", fontsize=16, fontweight="bold", pad=15)
plt.xlabel("Number of Articles", fontsize=10)
plt.yticks(fontsize=8)
plt.ylabel("")
sns.despine()
plt.show()


In [ ]:
# Pareto rule 

official_co = df[df["tier"] == "official-co"].sort_values(by="n_articles", ascending=False)

official_co["articles_cumsum"] = official_co["n_articles"].cumsum()

articles_cumsum = official_co["articles_cumsum"].iloc[-1] # Last value of the cumulative sum

official_co["articles_percentage"] = official_co["articles_cumsum"] / articles_cumsum * 100

pareto_official_co = official_co[official_co["articles_percentage"].shift(1, fill_value=0) < 80]


if pareto_official_co.shape[0] < 30:
    plt.figure(figsize=(10, 6))
    sns.barplot(data=pareto_official_co, x="n_articles", y="domain", palette="viridis", hue="n_articles", dodge=False, legend=False)
    bars = plt.gca().patches
    for bar in bars:
        width = bar.get_width()
        plt.gca().text(width + 1, bar.get_y() + bar.get_height()/2, f'{int(width)}', va='center', fontsize=8)

    plt.annotate(
        text=f"80% of articles are from\n{pareto_official_co.shape[0]} sources of {official_co.shape[0]} total sources",
        xy=(0.7, 0.1),
        xycoords="axes fraction",
        fontsize=10,
        bbox={'boxstyle': "round,pad=0.3", 'fc': "yellow", 'alpha': 0.5}
    )
    plt.title("Pareto Rule: Official-Co Sources with 80% of Articles", fontsize=16, fontweight="bold", pad=15)
    plt.xlabel("Number of Articles", fontsize=10)
    plt.yticks(fontsize=4)
    plt.ylabel("")
    sns.despine()
    plt.show()
else:
    print("Not graophing the Pareto rule for official-co sources because there are more than 30 sources.")
    n_sources = pareto_official_co.shape[0]
    print(f"80% of articles are from {n_sources} sources of {official_co.shape[0]} total sources.")

    sources = pareto_official_co["domain"].tolist()
    print(f"The sources are: {sources}")
    



In [ ]:
# Group the entities


def entity(d):
    p = d.split(".")
    return ".".join(p[-3:]) if d.endswith((".gov.co", ".mil.co")) else d

off = df[df["tier"] == "official-co"].copy()
off["entity"] = off["domain"].map(entity)

ent = off.groupby("entity", as_index=False).agg(
    n_articles=("n_articles", "sum"),
    n_links=("n_links", "sum"),
    n_domains=("domain", "size"),
).sort_values(by="n_articles", ascending=False)

ent["articles_cumsum"] = ent["n_articles"].cumsum()
articles_cumsum_ent = ent["articles_cumsum"].iloc[-1] # Last value of the cumulative sum
ent["articles_percentage"] = ent["articles_cumsum"] / articles_cumsum_ent * 100

pareto_ent = ent[ent["articles_percentage"].shift(1, fill_value=0) < 80]
pareto_ent
